In [156]:
import os
import random
from asgiref.sync import sync_to_async

from typing import cast

import logging
import tiktoken
import openai

from literev.models import Cluster, Document, ClusterElement
from literev.libs.nlp import call_chatgpt

from asgiref.sync import sync_to_async


In [157]:
logger = logging.getLogger(__name__)

API_KEY = "sk-proj-BW4s0WknbzkGoxpzaDHZT3BlbkFJnH2sD1NS1t7extu5sGVc"
GPT_MODEL = "gpt-4o-mini"
GPT_MODEL_MAX_TOKENS = 128000
MAX_TOKENS_RESPONSE = 250
# 200 tokens extra margin for security
TOKEN_LIMIT = GPT_MODEL_MAX_TOKENS - MAX_TOKENS_RESPONSE - 200
RANDOM_K = 27

In [158]:
# Func auxiliary to count the tokens using tiktoken library

In [159]:
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# num_tokens_from_string("tiktoken is great!", "o200k_base")


In [160]:
# Refactored function logging the tokens size in each step

In [165]:
def refactor_build_prompt(
    cluster: Cluster,
) -> str:
    """        
    #     # TODO : Create a formula ${3k * 27 = 81000 + 500 * 4/3} = 108667 tokens
    #     # 2114 * 27 = 57078 + 500 = 57578 * 4/3 = 76771 Tokens
    #     # Length Prompt:
    #     # 61434
      
    #     if i == 3:
    #         print(prompt + "\n".join(prompt_fragments) + prompt_constraints)
    #         # print(prompt_constraints)

     
    """

    RANDOM_K= 27
    prompt_fragments = []
    enc = tiktoken.encoding_for_model(GPT_MODEL)
    
    prompt_template = (
        "Provide a clear and concise general description based on the following "
        "most important keywords: {keywords} from a cluster containing "
        "law cases from the canton of Geneva in Switzerland. The summary must "
        "represent all law cases collectively and not be based only on a single one. "
        "In addition to the most important keywords, use "
        "the following additional context coming from the law cases contained in this cluster:"
        "\n"
    )

    prompt_constraints = (
        "Ensure the response maintains a narrative voice suitable for a general "
        "description while minimizing the use of pronouns. "
        "Avoid: unnecessary introductions and redundancies, using quotes, backticks, "
        "cluster's keywords and names of individuals, and words such as topic, case, "
        "we propose, proposing, we, this report. Summarize in exactly "
        "two sentences, ensuring no redundancy between sentences. "
        "Write the response in French."
    )
    
    document_context_template = (
        "{context}.\n"
    )
    
    prompt = prompt_template.format(keywords=cluster.topic)
    
    documents = Document.objects.filter(clusterelement__cluster=cluster)   

    for i, document in enumerate(documents, start=1):
        abstract_words = document.preprocessed_document.split()

        if len(abstract_words) > RANDOM_K:
            rnd_index = random.randrange(len(abstract_words) - RANDOM_K)
            document_abstract = " ".join(
                abstract_words[rnd_index : rnd_index + RANDOM_K]
            )
        else:
            logger.warning(
                f"Document {i} does not have enough abstract words."
            )
            document_abstract = " ".join(abstract_words)

        document_context = document_context_template.format(
            context=document_abstract
        )

        prompt_fragments.append(document_context)

        if (len(enc.encode(prompt + "".join(prompt_fragments)))) > TOKEN_LIMIT:
            logger.warning(
                "Reached token limit; stopping addition of further documents."
            )
            break  # Stop adding documents if token limit is exceeded

    full_prompt = prompt + "\n".join(prompt_fragments) + prompt_constraints

    print("\nLength Prompt:")

    total_tokens_encoded = num_tokens_from_string(full_prompt, "o200k_base")
    
    print(len(full_prompt.split()))
    
    
    print("total_tokens_encoded:", total_tokens_encoded)
    
    return full_prompt


In [166]:
# Using the django ORM inside a async process passing the prompt to the OpenAI as parameter 

In [167]:

@sync_to_async
def custom_nlp_summary(
    cluster_id: int,
) -> str:
    """"""
    if isinstance(cluster_id, int):
        cluster = Cluster.objects.get(id=cluster_id)


    prompt = refactor_build_prompt(
        cluster, 
    )

    
    # return call_chatgpt(   
    #     prompt=prompt, 
    #     api_key=API_KEY
    # )


In [168]:
cluster_id = 38

await  custom_nlp_summary(
    cluster_id, )

Document 922 does not have enough abstract words.
Document 1581 does not have enough abstract words.
Document 1892 does not have enough abstract words.
Document 2058 does not have enough abstract words.
Document 2066 does not have enough abstract words.
Document 2067 does not have enough abstract words.



Length Prompt:
57206
total_tokens_encoded: 89579


---

In [169]:
# Old func

In [140]:

def optimized_build_prompt(
    cluster: Cluster,
) -> str:
    """
    Build an optimized prompt for a scientific topic description.

    """
    random_k=27
    GPT_MODEL_MAX_TOKENS = 128000
    MAX_TOKENS_RESPONSE = 250
    
    TOKEN_LIMIT = GPT_MODEL_MAX_TOKENS - MAX_TOKENS_RESPONSE - 200

    print("randomk",  random_k)
    # GPT model max tokens (4096 for GPT-4o-mini)
    gpt_model_max_tokens = 128000

    # Calculate token budget
    token_limit = gpt_model_max_tokens - MAX_TOKENS_RESPONSE - 200

    prompt_template = (
        "Provide a clear and concise general description based on the following "
        "most important keywords: {keywords} from a cluster containing "
        "law cases from the canton of Geneva in Switzerland. The summary must "
        "represent all law cases collectively and not be based only on a single one. "
        "In addition to the most important keywords, use "
        "the following additional context coming from the law cases contained in this cluster:"
        "\n"
    )

    prompt_constraints = (
        "Ensure the response maintains a narrative voice suitable for a general "
        "description while minimizing the use of pronouns. "
        "Avoid: unnecessary introductions and redundancies, using quotes, backticks, "
        "cluster's keywords and names of individuals, and words such as topic, case, "
        "we propose, proposing, we, this report. Summarize in exactly "
        "two sentences, ensuring no redundancy between sentences. "
        "Write the response in French."
    )
    
    # Initialize tokenizer
    enc = tiktoken.encoding_for_model(gpt_model)

    # Inject keywords into prompt
    prompt = prompt_template.format(keywords=cluster.topic)
    
    # Template for document descriptions
    document_prompt_template = (
        "Document number: {document_number}\n"
        "Descriptor: {document_descriptor}\n"
        "Keywords Sample: {document_abstract}.\n\n"
    )

    prompt_fragments = []

    # Get all documents related to cluster(topic)
    documents = Document.objects.filter(clusterelement__cluster=cluster)

    # Iterate through documents to build the prompt
    for document_number, document in enumerate(documents, start=1):
        # Extract relevant details
        document_descriptor = document.descriptors
    
        abstract_words = document.preprocessed_document.split()
        if len(abstract_words) > random_k:
            rnd_index = random.randrange(len(abstract_words) - random_k)
            document_abstract = " ".join(abstract_words)
        else:
            document_abstract = document.preprocessed_document  # Fallback if not enough words
            print(f"Document {document_number} does not have enough abstract words.")

        document_abstract = (
            # get 27 random adjacent words from preprocessed text
            " ".join(
                abstract_words[rnd_index:rnd_index+random_k]
            )
            if len(abstract_words) > random_k
            else document.preprocessed_document
        )
        
        # Build document-specific prompt
        document_prompt = document_prompt_template.format(
            document_number=document_number,
            document_descriptor=document_descriptor,
            document_abstract=document_abstract,
        )

        # Count tokens for the document prompt
        document_tokens = len(enc.encode(document_prompt))

        # Check if the document fits within the remaining token budget
        # if document_tokens > TOKEN_LIMIT:
        # break  # Stop adding documents

        # Add document prompt to the main fragments
        prompt_fragments.append(document_prompt)
    
    # Finalize prompt with constraints
    full_prompt = prompt + "".join(prompt_fragments) + prompt_constraints

    prompt_end_full_tokens = len(enc.encode(full_prompt))
    base_tokens = len(enc.encode(prompt)) + len(enc.encode(prompt_constraints))

    print(
        "\n",f"prompt_tokens ({len(enc.encode(prompt))}):", prompt, "\n", f"prompt_constraints_tokens ({len(enc.encode(prompt_constraints))}):", prompt_constraints, "\n" )

    print("prompt_fragments_list:", len(enc.encode("".join(prompt_fragments))))
    print("prompt_end_full_tokens:", prompt_end_full_tokens)

    return full_prompt

In [141]:

@sync_to_async
def custom_nlp_origin_summary(
    cluster_id: int,
) -> str:
    """"""
    if isinstance(cluster_id, int):
        cluster = Cluster.objects.get(id=cluster_id)


    prompt = optimized_build_prompt(
        cluster, 
    )

    
    # return call_chatgpt(   
    #     prompt=prompt, 
    #     api_key=API_KEY
    # )


In [142]:
cluster_id = 38
gpt_model = "gpt-4o-mini"
API_KEY = "sk-proj-BW4s0WknbzkGoxpzaDHZT3BlbkFJnH2sD1NS1t7extu5sGVc"

await  custom_nlp_origin_summary(
    cluster_id, )

randomk 27
Document 922 does not have enough abstract words.
Document 1581 does not have enough abstract words.
Document 1892 does not have enough abstract words.
Document 2058 does not have enough abstract words.
Document 2066 does not have enough abstract words.
Document 2067 does not have enough abstract words.

 prompt_tokens (144): Provide a clear and concise general description based on the following most important keywords: fiscal, contribuable, impôt, taxation, réclamation, revenu, chambre, bordereau, administration, société, chf, imposable, icc, commission, montant, tapi, lifd, imposition, frais, ifd, déduction, jugement, immeuble, année, article, révision, instance, déclaration, amende, contribution from a cluster containing law cases from the canton of Geneva in Switzerland. The summary must represent all law cases collectively and not be based only on a single one. In addition to the most important keywords, use the following additional context coming from the law cases con